In [ ]:
import json, re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import numpy as np
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'


In [ ]:
# Set Chinese font for matplotlib so labels render correctly
import matplotlib
import matplotlib.font_manager as fm

# Try common CJK fonts present on macOS / Linux
_cjk_candidates = [
    "PingFang SC", "Heiti SC", "STHeiti", "SimHei",
    "WenQuanYi Zen Hei", "Noto Sans CJK SC",
]
_available = {f.name for f in fm.fontManager.ttflist}
_chosen = next((f for f in _cjk_candidates if f in _available), None)
if _chosen:
    matplotlib.rcParams["font.family"] = _chosen
    print(f"Using CJK font: {_chosen}")
else:
    print("No CJK font found — Chinese labels may render as boxes")
matplotlib.rcParams["axes.unicode_minus"] = False


In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
ax.set_aspect('equal')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

COLORS_PIE = {1: "#1A9E6A", 3: "#E67E22", 4: "#C0392B", 2: "#8E44AD", 5: "#2980B9"}
SPORTS_COLOR = "#1A6A9E"
COMM_COLOR   = "#7D3C98"

# Sports-specific cluster first, then commercial/financial — mirrors tree-graph grouping
topic_order  = [1, 3, 4, 2, 5]
pct_d        = {1: 41.48, 3: 15.17, 4: 13.52, 2: 23.35, 5: 6.48}
pcts         = [pct_d[t] for t in topic_order]
wedge_labels = [f"T{t}\n{pct_d[t]:.1f}%" for t in topic_order]

wedges, texts = ax.pie(
    pcts,
    labels=wedge_labels,
    colors=[COLORS_PIE[t] for t in topic_order],
    startangle=90,
    labeldistance=1.10,
    wedgeprops=dict(edgecolor='white', linewidth=2.5, width=0.52),
    textprops=dict(fontsize=8.5, fontweight='bold'),
)
for text, tn in zip(texts, topic_order):
    text.set_color(COLORS_PIE[tn])

ax.text(0, 0, 'Athlete\nCourt Cases\n(n = 79)',
        ha='center', va='center', fontsize=10.5,
        fontweight='bold', color='#2C3E50')

# --- external category arcs ---
# wedges 0–2 = sports-specific (T1, T3, T4); wedges 3–4 = commercial (T2, T5)
sports_t1 = wedges[0].theta1     # 90° (top)
sports_t2 = wedges[2].theta2     # end of T4  ~342.6°
comm_t1   = wedges[3].theta1     # start of T2  ~342.6°
comm_t2   = wedges[4].theta2     # end of T5  ~450° (= 90° + 360°)

ARC_INNER, ARC_OUTER, GAP = 1.38, 1.50, 3.0

ax.add_patch(mpatches.Wedge(
    (0, 0), ARC_OUTER, sports_t1 + GAP, sports_t2 - GAP,
    width=ARC_OUTER - ARC_INNER, color=SPORTS_COLOR, alpha=0.90, edgecolor='none',
))
ax.add_patch(mpatches.Wedge(
    (0, 0), ARC_OUTER, comm_t1 + GAP, comm_t2 - GAP,
    width=ARC_OUTER - ARC_INNER, color=COMM_COLOR,   alpha=0.90, edgecolor='none',
))

# arc category text labels
LABEL_R    = 1.70
sports_mid = np.radians((sports_t1 + sports_t2) / 2)
comm_mid   = np.radians((comm_t1   + comm_t2)   / 2)

ax.text(LABEL_R * np.cos(sports_mid), LABEL_R * np.sin(sports_mid),
        'Sports-Specific\nDisputes',
        ha='center', va='center', fontsize=9.5, fontweight='bold', color=SPORTS_COLOR)
ax.text(LABEL_R * np.cos(comm_mid), LABEL_R * np.sin(comm_mid),
        'Commercial /\nFinancial Disputes',
        ha='center', va='center', fontsize=9.5, fontweight='bold', color=COMM_COLOR)

ax.set_xlim(-2.1, 2.1)
ax.set_ylim(-2.2, 2.1)
ax.axis('off')

legend_handles = [
    mpatches.Patch(color=COLORS_PIE[1], label='T1 – Athlete Retirement Employment Compensation  (41.5%)'),
    mpatches.Patch(color=COLORS_PIE[3], label='T3 – Sports Injury & Medical Compensation  (15.2%)'),
    mpatches.Patch(color=COLORS_PIE[4], label='T4 – Image Rights & Online Infringement  (13.5%)'),
    mpatches.Patch(color=COLORS_PIE[2], label='T2 – Commercial Contract Breach  (23.4%)'),
    mpatches.Patch(color=COLORS_PIE[5], label='T5 – Debt, Lending & Recovery  (6.5%)'),
]
ax.legend(handles=legend_handles, loc='lower center', ncol=1,
          fontsize=8, framealpha=0.85, edgecolor='#ccc',
          bbox_to_anchor=(0.5, -0.01))

ax.set_title(
    'LDA Topic Distribution — Athlete Court Cases\n'
    '(outer arc = dispute category; slice area ∝ corpus share)',
    fontsize=11, fontweight='bold', pad=14,
)
plt.tight_layout()
plt.savefig('../output/topic_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → ../output/topic_pie.png")